# Cas d'usage certif — Orientation et tri multimodal des demandeurs d'emploi


# 0 Imports et constantes

In [11]:
import hashlib, json, platform, random, subprocess, sys
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42                      # une seule graine, partout
random.seed(SEED)
np.random.seed(SEED)

DATA_PATH = Path("../../dataset_trajectoire_emploi_Sujet.csv")
OUTPUT_DIR = Path("../../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET = "classe_retour_emploi"
CLASS_LABELS = {0: "Rapide (< 6 mois)", 1: "Moyen (6-12 mois)", 2: "Longue durée (> 12 mois)"}

---
# 1 Cadrage

**Objectif** : poser par écrit le besoin, la tâche, les métriques et les risques **avant** de toucher aux données.

## 1.1 Besoin métier

Une agence nationale de l'emploi doit **aiguiller rapidement** les demandeurs d'emploi à l'issue du premier entretien. L'enjeu : repérer très tôt les usagers à risque de **chômage de longue durée** pour leur proposer un accompagnement renforcé, sans mobiliser ce dispositif pour ceux qui retrouveront un emploi rapidement.

L'outil est une **aide à la décision** pour le conseiller (qui garde la main), pas une décision automatique.

> **Ce que dit le sujet** : « repérer très tôt les usagers à risque de chômage de longue durée » pour un « aiguillage rapide et fiable ». **Ce qu'on en comprend** : il ne s'agit pas de remplacer la décision du conseiller mais de lui fournir un signal de risque priorisé, avant l'entretien ou juste après, pour l'aider à arbitrer *qui* mérite un accompagnement renforcé — l'IA ne doit jamais décider seule (cf. RGPD art. 22).

## 1.2 Tâche ML

- **Apprentissage supervisé**, classification **multiclasse ordonnée** (0 < 1 < 2) : 0 = retour rapide (< 6 mois), 1 = moyen (6-12 mois), 2 = longue durée (> 12 mois).
- **Données hybrides** : tabulaires (âge, diplôme, ancienneté, ROME, INSEE, allocataire, nationalité) + texte libre (synthèse du conseiller).
- **2 500 échantillons**, 9 variables explicatives + 1 identifiant + la cible.

## 1.2bis Cartographie de la source

Une seule source de données ici (pas de multi-sources à croiser) : le tableau ci-dessous en fait l'inventaire en une ligne, façon fiche `302_Cartographie_sources`.

| Source | Format | Volume | Fréquence | Qualité observée à l'œil nu | Risques RGPD | Pertinence métier |
|---|---|---|---|---|---|---|
| `dataset_trajectoire_emploi_Sujet.csv` | CSV | 2 500 lignes × 10 colonnes | **snapshot unique** (pas de flux, pas de mise à jour prévue) | manquants sur 4 colonnes (~2-5 %), 74 lignes incohérentes (ancienneté > âge − 15), 1 code ROME atypique (W1401) | Variable sensible directe (`nationalite_hors_ue`), quasi-identifiant géographique (`code_insee_commune`), texte libre à PII potentielles (`synthese_entretien`) | **Totale** — c'est l'unique source, pas d'écart possible |

## 1.3 Données et points de vigilance sur l'étiquette

| Point | Pourquoi c'est important |
|---|---|
| La cible est « la décision de parcours **ou** le constat terrain validé par les conseillers » | Mélange d'une **décision** et d'un **résultat observé**. Un modèle entraîné sur des décisions **imite** les conseillers (biais compris) au lieu de prédire un retour à l'emploi réel. |
| La synthèse est rédigée **après** l'entretien, probablement par la personne qui fixe l'étiquette | Risque de **contamination** : le texte peut refléter l'issue, pas seulement l'état de départ. La performance du scénario texte seul sera peut-être surestimée. |
| Aucune date, aucun identifiant de conseiller ou d'agence | Pas de découpage temporel possible ; pas de contrôle de fuite par groupe. La Cross-Validation aléatoire est peut-être optimiste. |

## 1.4 Métriques et coût de l'erreur

| Métrique | Rôle |
|---|---|
| Accuracy | Référence lisible (mais insuffisante : classes déséquilibrées) |
| **F1 macro** | Métrique principale de comparaison (chaque classe pèse pareil) |
| Matrice de confusion | Voir *où* le modèle se trompe |
| **Recall de la classe 2** | Part des usagers à risque réellement détectés |
| **Taux d'erreur critique 2→0** | Usager à risque classé « retour rapide » = perte d'accompagnement (erreur la plus grave selon le sujet) |
| Kappa pondéré quadratique | Tient compte de l'ordre des classes (2→0 est pire que 2→1) |
| Coût moyen par usager | Synthèse sous **matrice de coûts** (hypothèse ci-dessous, à valider avec le métier) |

## 1.5 Point d'étape éthique n°1 — dictionnaire des risques

| Variable | Nature du risque | Références à vérifier | Position provisoire |
|---|---|---|---|
| `usager_id` | Identifiant direct | RGPD art. 4 (pseudonymisation) | Exclu des features ; conservé pour la traçabilité |
| `age` | Critère discriminatoire ; **très prédictif** | Code pénal art. 225-1 ; Code du travail L1132-1 | **À arbitrer** : gardé en S1 ; à débattre en S2 selon le poids de la colonne dans la prédiction. |
| `niveau_diplome` | Socio-professionnel, non protégé en soi | — | Conservé |
| `anciennete_poste_ans` | Idem ; valeurs incohérentes à nettoyer | — | Conservé après nettoyage |
| `code_rome_vise` | Projet professionnel ; **signal réel** | — | Conservé |
| `code_insee_commune` | Lieu de résidence (critère discriminatoire, proxy socio-territorial) ; haute cardinalité ; quasi-identifiant | Code pénal art. 225-1 ; RGPD minimisation | **À arbitrer** ; agréger en département (Corse 2A/2B, DOM sur 3 chiffres) ou retirer |
| `est_allocataire` | Situation économique (vulnérabilité) ; **aucun signal** au premier examen | Code pénal art. 225-1 ; RGPD minimisation | Candidat à la suppression |
| `nationalite_hors_ue` | Nationalité, **proxy de l'origine** ; **signal le plus fort du jeu** | Code pénal art. 225-1 ; RGPD art. 9 (proxy, à discuter) | **Retiré en S2** ; utilisé seulement pour l'**audit d'équité** (base légale à valider avec le DPO) |
| `synthese_entretien` | Texte libre : PII et données sensibles possibles (santé, famille) ; **proxies** (« barrière de la langue », « garde d'enfants ») | RGPD art. 9 ; art. 225-1 | Scénario 3 pour test ; audit des proxies |
| `classe_retour_emploi` | Cible : décision vs constat ; biais historique possible | RGPD art. 22 ; CRPA art. L311-3-1 | Documenter ; humain dans la boucle |


> ⚠️ **Ré-identification indirecte (fiche `303_Risques_RGPD_multisources`)** : `code_insee_commune` seul ne révèle rien, mais **croisé** avec `age` et `niveau_diplome` sur une commune à faible population, la combinaison peut redevenir quasi-unique et donc ré-identifiante — même sans variable sensible directe. C'est un argument de plus pour agréger en département plutôt que garder le code commune brut en feature.

**Cadre général à traiter dans le dossier** :
- **RGPD** : minimisation, finalité, AIPD (art. 35), décision assistée avec intervention humaine effective (art. 22) ;
- **Loi pour une République Numérique** : explicabilité des décisions algorithmiques publiques ;
- **Règlement européen sur l'IA** : cas d'usage probablement à **haut risque** (emploi / accès aux services essentiels) ;
- **Responsabilité** : qui répond d'une erreur d'orientation qui prive l'usager d'un accompagnement (perte de chance) ?

**Questions à trancher** :
1. Le scénario 2 retire-t-il **seulement la nationalité**, ou aussi âge / INSEE / allocataire ?
2. Que fait-on des **proxys dans le texte** ?
3. La nationalité sert-elle à l'audit d'équité ? Sur quelle base légale ?

## 1.6 Périmètre

| Dans le périmètre | Hors périmètre |
|---|---|
| 4 scénarios (S1 complet · S2 sans variables sensibles · S3 texte seul · S4 tabulaire seul) | LLM / agents / RAG « parce que c'est possible » |
| Familles : régression logistique, Random Forest, LightGBM, XGBoost | Deep learning, embeddings |
| Décision à coût minimal, calibration, audit d'équité | Décision entièrement automatique |
| Industrialisation | Docker / CI-CD / MLflow |

**Condition de changement d'avis : on reste sur du TF-IDF classique pour le texte (ML classique, coût le plus bas) tant que le F1 macro du scénario S3 (texte seul) ne s'effondre pas anormalement par rapport aux scénarios tabulaires. On envisagerait des embeddings/transformers pré-entraînés *seulement* si ce diagnostic le justifie à l'Étape 5 — jamais de LLM génératif, hors périmètre du sujet dans tous les cas.

**Traçage des expériences : chaque run de modèle (à partir de l'Étape 5) sera consigné dans un `experiments.md` versionné (nom, hyperparamètres, split, métriques, verdict retenu/écarté).

---
# 2 Exploration

**Objectif** : vérifier à l'œil ce qu'on a réellement sous la main (qualité, cible, cardinalités, signal éthique), avant de préparer le pipeline.

In [12]:
import sys
sys.path.insert(0, str(Path("../src").resolve()))

df = pd.read_csv(DATA_PATH, dtype={"code_insee_commune": "string"})
print(f"shape : {df.shape}")
df.head(3)  # aperçu limité à 3 lignes (règle §1.3 AGENTS.md)


shape : (2500, 10)


,usager_id,age,niveau_diplome,anciennete_poste_ans,code_rome_vise,code_insee_commune,est_allocataire,nationalite_hors_ue,synthese_entretien,classe_retour_emploi
0,ID_0000,56.0,Bac+2,2.6,D1503,18273,1.0,0,Compétences à réactualiser sur les outils numé...,1
1,ID_0001,46.0,Sans diplôme,10.5,G1302,07240,0.0,0,Cumul de difficultés. Pas de moyen de transpor...,2
2,ID_0002,NaN,Bac,1.4,M1705,01405,NaN,0,Compétences à réactualiser sur les outils numé...,0


## 2.1 Audit qualité


In [13]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   usager_id             2500 non-null   object 
 1   age                   2378 non-null   float64
 2   niveau_diplome        2416 non-null   object 
 3   anciennete_poste_ans  2500 non-null   float64
 4   code_rome_vise        2500 non-null   object 
 5   code_insee_commune    2500 non-null   string 
 6   est_allocataire       2456 non-null   float64
 7   nationalite_hors_ue   2500 non-null   int64  
 8   synthese_entretien    2419 non-null   object 
 9   classe_retour_emploi  2500 non-null   int64  
dtypes: float64(3), int64(2), object(4), string(1)
memory usage: 195.4+ KB


In [14]:
manquants = (df.isna().mean() * 100).round(2).sort_values(ascending=False)
print("% de manquants par colonne :")
print(manquants)
print()
print("doublons usager_id :", df["usager_id"].duplicated().sum())
print()
print("distribution de la cible (%) :")
print((df["classe_retour_emploi"].value_counts(normalize=True) * 100).round(1).sort_index())


% de manquants par colonne :
age                     4.88
niveau_diplome          3.36
synthese_entretien      3.24
est_allocataire         1.76
usager_id               0.00
anciennete_poste_ans    0.00
code_rome_vise          0.00
code_insee_commune      0.00
nationalite_hors_ue     0.00
classe_retour_emploi    0.00
dtype: float64

doublons usager_id : 0

distribution de la cible (%) :
classe_retour_emploi
0    37.4
1    44.5
2    18.1
Name: proportion, dtype: float64


**Constat** : les pourcentages de manquants et la distribution de la cible confirment exactement les chiffres du cadrage : `age` ≈ 4,88 %, `niveau_diplome` ≈ 3,36 %, `est_allocataire` ≈ 1,76 %, `synthese_entretien` ≈ 3,24 %, cible ≈ 37,4 / 44,5 / 18,1 %. Aucun doublon d'`usager_id`.


## 2.2 Incohérences et cardinalités


In [15]:
incoherentes = df[df["anciennete_poste_ans"] > (df["age"] - 15)]
print(f"lignes incohérentes (ancienneté > âge - 15) : {len(incoherentes)}")
print()

lettres_rome = df["code_rome_vise"].str[0].value_counts().sort_index()
print("lettres de code ROME rencontrées :")
print(lettres_rome)
print()
print(f"cardinalité code_rome_vise : {df['code_rome_vise'].nunique()}")
print(f"cardinalité code_insee_commune : {df['code_insee_commune'].nunique()}")
print(f"codes commençant par 2A/2B (Corse) : {df['code_insee_commune'].str.startswith(('2A', '2B')).sum()}")
print(f"codes DOM (971-976) : {df['code_insee_commune'].str.startswith(('971','972','973','974','976')).sum()}")


lignes incohérentes (ancienneté > âge - 15) : 74

lettres de code ROME rencontrées :
code_rome_vise
A    251
C    240
D    248
F     38
G    257
H    180
I     62
J     96
K    392
M    455
N    251
W     30
Name: count, dtype: int64

cardinalité code_rome_vise : 50
cardinalité code_insee_commune : 2407
codes commençant par 2A/2B (Corse) : 21
codes DOM (971-976) : 10


**Constat** :
- **74 lignes** incohérentes confirmées (ancienneté > âge − 15) → décision D4 toujours ouverte (corriger / exclure / marquer), à trancher en Étape 3.
- La nomenclature ROME officielle va de **A à N** (14 domaines). Le code `W1401` (30 occurrences) utilise la lettre **W, absente de la nomenclature réelle** — il s'agit de l'écart mentionné plus bas : un **code non conforme** à vérifier (bug de génération du jeu synthétique, à mentionner dans les limites).
- `code_insee_commune` : cardinalité 2 407 (quasi une par ligne), 21 codes Corse (2A/2B), 10 codes DOM — confirme la nécessité d'agréger en département plutôt que garder le code brut.


## 2.3 Texte libre — vérification PII


In [16]:
phrases = df["synthese_entretien"].value_counts(dropna=True)
print(f"nombre de phrases distinctes : {df['synthese_entretien'].dropna().nunique()}")
phrases  # value_counts = agrégat, pas des lignes individuelles


nombre de phrases distinctes : 9


synthese_entretien
Compétences à réactualiser sur les outils numériques. Motivation correcte.       338
Problème ponctuel de mobilité géographique. Zone mal desservie.                  334
Profil autonome, recherche active, aucun frein périphérique identifié.           300
Projet de reconversion à affiner. Besoin d'une formation complémentaire.         275
Excellente présentation, compétences techniques à jour. Reprise rapide visée.    272
Candidat très dynamique, mobilité nationale validée. Projet clair.               259
Cumul de difficultés. Pas de moyen de transport et garde d'enfants complexe.     227
Perte de confiance importante. Risque d'exclusion, barrière de la langue.        212
Freins périphériques majeurs. Situation d'illettrisme numérique constatée.       202
Name: count, dtype: int64

**Constat** : seulement **9 phrases-gabarits** distinctes, aucune ne contient de nom, coordonnée ou identifiant — le texte est manifestement **synthétique et templaté**, pas rédigé au cas par cas. **Décision** : Les proxys socio-culturels repérés dans certaines phrases (« barrière de la langue », « garde d'enfants ») restent un point d'audit éthique, mais pas un risque de ré-identification.


## 2.4 Disparate impact — premier diagnostic

Réutilisation du module `src/trajectoire_emploi/fairness.py` (testé dans `tests/test_fairness.py`). **Outcome positif retenu : classe 0 (retour rapide)**


In [17]:
from trajectoire_emploi.fairness import disparate_impact, verdict_4_5, supports_fiables

di, sr = disparate_impact(
    df, "nationalite_hors_ue", "classe_retour_emploi",
    est_positif=lambda s: s == 0,
)
print("taux de retour rapide par groupe :")
print(sr)
print(f"\nDI = {di:.3f} — {verdict_4_5(di)}")
print()
print("effectifs par groupe :")
print(df["nationalite_hors_ue"].value_counts())
print(supports_fiables(df["nationalite_hors_ue"].value_counts()))


taux de retour rapide par groupe :
nationalite_hors_ue
0    0.405896
1    0.138983
Name: classe_retour_emploi, dtype: float64

DI = 0.342 — ⚠️ signal (DI < 0.8) — groupe minoritaire désavantagé

effectifs par groupe :
nationalite_hors_ue
0    2205
1     295
Name: count, dtype: int64
Series([], Name: count, dtype: int64)


**Constat** : DI ≈ **0,342**, très en dessous du seuil de 0,8 → **signal majeur** confirmant que `nationalite_hors_ue` est « le signal le plus fort du jeu » (§3.2). Les usagers de nationalité hors UE ont un taux de retour rapide de 13,9 % contre 40,6 % pour les autres — écart qui ne prouve pas une discrimination du futur modèle (cf. avertissement fiche 202) mais documente un **biais déjà présent dans les données historiques**, à surveiller de près en scénario S1 et à retirer en S2.
Effectif du groupe minoritaire (295) largement au-dessus du seuil de support fiable de 30 (fiche 211) : le signal n'est pas un artefact de petit échantillon.


## 2.5 Datasheet Gebru

Brouillon des 7 sections, à étoffer/corriger avec vous avant validation finale (ce n'est **pas** un livrable figé) :

**1. Motivation** — Jeu synthétique construit pour la certification « Orientation et tri multimodal des demandeurs d'emploi » ; simule des données d'agence nationale de l'emploi à l'issue du premier entretien.

**2. Composition** — 2 500 lignes × 10 colonnes (9 features + cible). Cible ordinale à 3 classes (37,4 / 44,5 / 18,1 %). Variable sensible directe : `nationalite_hors_ue` (DI ≈ 0,342, signal majeur — cf. 2.4). Quasi-identifiant : `code_insee_commune` (haute cardinalité, risque de croisement — cf. 1.5). Pas de PII textuelle détectée dans `synthese_entretien` (cf. 2.3).

**3. Processus de collecte** — Inconnu / **jeu manifestement synthétique** (cf. code ROME `W1401` non conforme à la nomenclature réelle) — à mentionner explicitement dans les limites du projet.

**4. Preprocessing appliqué** — Aucun à ce stade (Étape 2 = exploration seule, rien n'est encore modifié dans les données).

**5. Usages prévus / à éviter** — Prévu : aide à la décision pour le conseiller (humain dans la boucle). À éviter : décision automatisée sans intervention humaine (RGPD art. 22), usage de `nationalite_hors_ue` ailleurs que dans l'audit d'équité (cf. décision D5, encore ouverte).

**6. Distribution** — Usage interne au projet de certification, pas de redistribution.

**7. Maintenance** — À compléter (auteur, version, date) une fois le document validé par vous.


## 2.6 Dictionnaire de variables


| Colonne | Type | Sensible ? | Remarque |
|---|---|---|---|
| `usager_id` | identifiant | Non (mais exclu des features) | 0 doublon |
| `age` | numérique | Non directement (proxy possible) | 4,88 % manquants |
| `niveau_diplome` | catégorielle (4 modalités) | Non | 3,36 % manquants |
| `anciennete_poste_ans` | numérique | Non | 74 lignes incohérentes vs `age` |
| `code_rome_vise` | catégorielle (50 modalités) | Non | code `W1401` non conforme (nomenclature A-N) |
| `code_insee_commune` | catégorielle haute cardinalité | **Oui (quasi-identifiant)** | 2 407 modalités, Corse/DOM à traiter |
| `est_allocataire` | binaire | Non (situation économique) | 1,76 % manquants, aucun signal DI au premier examen |
| `nationalite_hors_ue` | binaire | **Oui (sensible directe)** | DI ≈ 0,342, signal le plus fort |
| `synthese_entretien` | texte libre | Non (pas de PII détectée) | 9 phrases-gabarits seulement, 3,24 % manquants |
| `classe_retour_emploi` | cible ordinale (0/1/2) | — (cible) | mélange décision/constat, cf. §1.3 |
